In [1]:
#---------------------------подключение к базе данных -------------------------------
import os

from sqlalchemy import create_engine, text
from dotenv import load_dotenv


def get_engine():
    """Создаёт engine один раз. Проверяет подключение, но пул НЕ закрывает."""
    load_dotenv()
    host = os.getenv("DB_DESTINATION_HOST")
    port = os.getenv("DB_DESTINATION_PORT")
    db_name = os.getenv("DB_DESTINATION_NAME")
    username = os.getenv("DB_DESTINATION_USER")
    password = os.getenv("DB_DESTINATION_PASSWORD")
    if not all([host, port, db_name, username, password]):
        raise ValueError("Не все переменные окружения для подключения к БД заданы")

    dsn = (
        f"postgresql+psycopg2://{username}:{password}"
        f"@{host}:{port}/{db_name}?sslmode=disable"
    )
    engine = create_engine(dsn)

    # Проверяем подключение (соединение вернётся в пул, пул останется живым)
    try:
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        print(f"✅ Успешное подключение к базе данных '{db_name}' на {host}:{port}")
    except Exception as e:
        print(f"❌ Не удалось подключиться к базе данных: {e}")
        engine.dispose()
        raise

    return engine


engine = get_engine()

✅ Успешное подключение к базе данных 'mle_20261022_933a71d556bb' на pg.plg.education-services.ru:5432


In [65]:
#--------------------------------закрытие всех подключений-----------------------
import os
import psycopg2
from dotenv import load_dotenv

# Загружаем переменные из файла .env (если он есть в папке)
load_dotenv()

# Получаем данные из переменных окружения
host = os.getenv("DB_DESTINATION_HOST")
port = os.getenv("DB_DESTINATION_PORT", "5432") # 5432 - значение по умолчанию
db_name = os.getenv("DB_DESTINATION_NAME")
username = os.getenv("DB_DESTINATION_USER")
password = os.getenv("DB_DESTINATION_PASSWORD")

# Проверка, что все переменные заданы
if not all([host, db_name, username, password]):
    raise ValueError("Не все переменные окружения для подключения к БД заданы!")

try:
    # Подключаемся к базе
    conn = psycopg2.connect(
        host=host,
        port=str(port),
        dbname=db_name,
        user=username,
        password=password
    )
    conn.autocommit = True

    # Завершаем все сторонние подключения к этой базе
    with conn.cursor() as cur:
        cur.execute(f"""
            SELECT pg_terminate_backend(pid)
            FROM pg_stat_activity
            WHERE datname = '{db_name}'
              AND pid <> pg_backend_pid();
        """)
        # Получаем результат, чтобы убедиться, что запрос прошел
        terminated = cur.fetchall()
        print(f"Успешно завершено подключений: {len(terminated)}")
        print("Все сторонние подключения к базе данных освобождены.")

except Exception as e:
    print(f"Произошла ошибка: {e}")

finally:
    # Обязательно закрываем соединение, если оно было создано
    if 'conn' in locals() and conn:
        conn.close()

Произошла ошибка: connection to server at "pg.plg.education-services.ru" (138.16.201.121), port 5432 failed: FATAL:  too many connections for database "mle_20261022_933a71d556bb"



In [3]:
#-------------------таблицы в DB_DESTINATION_USER---------------------------------
def list_tables(engine, schema=None):
    """Выводит список таблиц и представлений (без системных схем)."""
    query = text("""
        SELECT table_schema, table_name, table_type
        FROM information_schema.tables
        WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
          AND (:schema IS NULL OR table_schema = :schema)
        ORDER BY table_schema, table_name
    """)
    with engine.connect() as conn:
        rows = conn.execute(query, {"schema": schema}).fetchall()

    if not rows:
        print("⚠️ Таблиц не найдено.")
        return rows

    print(f"📋 Найдено объектов: {len(rows)}\n")
    print(f"{'СХЕМА':<20} {'ТАБЛИЦА':<40} {'ТИП'}")
    print("-" * 90)
    for schema_name, table_name, table_type in rows:
        print(f"{schema_name:<20} {table_name:<40} {table_type}")
    return rows


def find_table(engine, name_pattern):
    """Ищет таблицы по частичному совпадению имени (без учёта регистра)."""
    query = text("""
        SELECT table_schema, table_name, table_type
        FROM information_schema.tables
        WHERE table_name ILIKE :pattern
        ORDER BY table_schema, table_name
    """)
    with engine.connect() as conn:
        rows = conn.execute(query, {"pattern": f"%{name_pattern}%"}).fetchall()

    print(f"\n🔎 Поиск по шаблону '%{name_pattern}%': найдено {len(rows)}")
    for schema_name, table_name, table_type in rows:
        print(f"   {schema_name}.\"{table_name}\"  ({table_type})")
    return rows

In [4]:
#-------------------таблицы в DB_DESTINATION_USER---------------------------------
list_tables(engine)

# Только схема public (раскомментируйте при необходимости)
# list_tables(engine, schema="public")

📋 Найдено объектов: 67

СХЕМА                ТАБЛИЦА                                  ТИП
------------------------------------------------------------------------------------------
public               alembic_version                          BASE TABLE
public               assessments                              BASE TABLE
public               budget_policies                          BASE TABLE
public               clean_users_churn                        BASE TABLE
public               datasets                                 BASE TABLE
public               endpoint_bindings                        BASE TABLE
public               endpoint_model_mappings                  BASE TABLE
public               endpoint_tags                            BASE TABLE
public               endpoints                                BASE TABLE
public               entity_associations                      BASE TABLE
public               evaluation_dataset_records               BASE TABLE
public          

[('public', 'alembic_version', 'BASE TABLE'),
 ('public', 'assessments', 'BASE TABLE'),
 ('public', 'budget_policies', 'BASE TABLE'),
 ('public', 'clean_users_churn', 'BASE TABLE'),
 ('public', 'datasets', 'BASE TABLE'),
 ('public', 'endpoint_bindings', 'BASE TABLE'),
 ('public', 'endpoint_model_mappings', 'BASE TABLE'),
 ('public', 'endpoint_tags', 'BASE TABLE'),
 ('public', 'endpoints', 'BASE TABLE'),
 ('public', 'entity_associations', 'BASE TABLE'),
 ('public', 'evaluation_dataset_records', 'BASE TABLE'),
 ('public', 'evaluation_dataset_tags', 'BASE TABLE'),
 ('public', 'evaluation_datasets', 'BASE TABLE'),
 ('public', 'experiment_tags', 'BASE TABLE'),
 ('public', 'experiments', 'BASE TABLE'),
 ('public', 'guardrail_configs', 'BASE TABLE'),
 ('public', 'guardrails', 'BASE TABLE'),
 ('public', 'input_tags', 'BASE TABLE'),
 ('public', 'inputs', 'BASE TABLE'),
 ('public', 'issues', 'BASE TABLE'),
 ('public', 'job_locks', 'BASE TABLE'),
 ('public', 'jobs', 'BASE TABLE'),
 ('public', 'la

In [34]:
find_table(engine, "users_charn")


🔎 Поиск по шаблону '%users_charn%': найдено 0


[]

In [5]:
import pandas as pd

with engine.connect() as conn:
    df_tables = pd.read_sql(
        text("""
            SELECT table_schema, table_name, table_type
            FROM information_schema.tables
            WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
            ORDER BY table_schema, table_name
        """),
        con=conn,
    )
df_tables

,table_schema,table_name,table_type
0,public,alembic_version,BASE TABLE
1,public,assessments,BASE TABLE
2,public,budget_policies,BASE TABLE
3,public,clean_users_churn,BASE TABLE
4,public,datasets,BASE TABLE
...,...,...,...
62,public,trace_tags,BASE TABLE
63,public,users_churn,BASE TABLE
64,public,webhook_events,BASE TABLE
65,public,webhooks,BASE TABLE


In [55]:
engine.dispose()
print("Пул соединений освобождён")

Пул соединений освобождён


In [ ]:
#----------------------------сколько соединений к базе DB_DESTINATION_USER------------------
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

# Подключение
dsn = f"postgresql+psycopg2://{os.getenv('DB_DESTINATION_USER')}:{os.getenv('DB_DESTINATION_PASSWORD')}@{os.getenv('DB_DESTINATION_HOST')}:{os.getenv('DB_DESTINATION_PORT')}/{os.getenv('DB_DESTINATION_NAME')}?sslmode=disable"
engine = create_engine(dsn, pool_pre_ping=True, pool_recycle=300)

# 1. Сводка: кто и сколько держит соединений
sql = text("""
    SELECT state, application_name, count(*) as connections, max(now() - state_change) as max_age
    FROM pg_stat_activity
    WHERE datname = current_database()
    GROUP BY state, application_name
    ORDER BY connections DESC;
""")

with engine.connect() as conn:
    display(pd.DataFrame(conn.execute(sql).fetchall(), columns=["state", "app", "count", "max_age"]))
    
    # 2. Лимит базы
    limit = conn.execute(text("SELECT datconnlimit FROM pg_database WHERE datname = current_database();")).scalar()
    print(f"Лимит (datconnlimit): {limit}")

In [ ]:
#-------------------------убить соединения к базе DB_DESTINATION_USER висящие больше 10 минут------------------------
with engine.begin() as conn:
    # Убиваем только "спящие" соединения, висящие больше 10 минут
    killed = conn.execute(text("""
        SELECT pg_terminate_backend(pid)
        FROM pg_stat_activity
        WHERE datname = current_database()
          AND state = 'idle'
          AND pid <> pg_backend_pid()
          AND now() - state_change > interval '10 minutes';
    """)).fetchall()
    
print(f"✅ Завершено соединений: {len(killed)}")

In [2]:
# 📌 Получаем список всех таблиц в БД
from sqlalchemy import create_engine, inspect

inspector = inspect(engine)
tables = inspector.get_table_names()
print("📌 Таблицы в базе данных:")
for t in tables:
    print(" •", t)
    
    
# 📌 Получаем список всех таблиц в БД с колонками
inspector = inspect(engine)
tables = inspector.get_table_names()
target = "clean_users_churn"
print("TABLE NAME:", target)
for schema in inspector.get_schema_names():
    if target in inspector.get_table_names(schema=schema):
        cols = inspector.get_columns(target, schema=schema)
        print("FOUND IN SCHEMA:", schema)
        print([c["name"] for c in cols])
        break
else:
    print("TABLE NOT FOUND:", target)

📌 Таблицы в базе данных:
 • users_churn
 • datasets
 • input_tags
 • clean_users_churn
 • trace_tags
 • trace_request_metadata
 • runs
 • tags
 • params
 • metrics
 • logged_models
 • alembic_version
 • experiment_tags
 • latest_metrics
 • logged_model_metrics
 • logged_model_params
 • logged_model_tags
 • inputs
 • entity_associations
 • webhook_events
 • scorers
 • scorer_versions
 • evaluation_dataset_tags
 • evaluation_dataset_records
 • endpoint_tags
 • trace_metrics
 • endpoint_model_mappings
 • online_scoring_configs
 • endpoint_bindings
 • span_metrics
 • experiments
 • registered_models
 • model_versions
 • registered_model_tags
 • model_version_tags
 • registered_model_aliases
 • evaluation_datasets
 • webhooks
 • secrets
 • endpoints
 • model_definitions
 • issues
 • guardrails
 • guardrail_configs
 • workspaces
 • label_schemas
 • review_queues
 • review_queue_users
 • review_queue_items
 • review_queue_label_schemas
 • mcp_servers
 • mcp_server_versions
 • mcp_server_tags
